📂 Project: AI-Adversarial Trace Analyzer (Nemesis Prototype)
🎯 Objective
To analyze high-fidelity synthetic adversarial traces to identify where automated Endpoint Detection and Response (EDR) systems fail. This project serves as the Knowledge Base for a future AI Security Analyst Agent.

🛠️ Tech Stack & Skills
Data Science: Python, Pandas, JSON Normalization.

Data Engineering: Processing Large-Scale Parquet files (2.5M+ rows).

Cybersecurity: Adversarial simulation analysis (MCTS reasoning), Red Team strategy identification.

🔍 What I Did Today (May 15, 2026)
Environment Setup: Initialized a Kaggle Notebook and connected the Nemesis Cyber Threat Simulation Pack.

Data Ingestion: Successfully loaded the simulation data using pd.read_parquet.

Adversarial Extraction: Flattened the nested agent_reasoning JSON data to reveal attacker confidence scores and "Winning Strategies."

Gap Analysis: Filtered the dataset to isolate 1,999,949 instances where the attack bypassed the EDR and was only caught by the SOC (or not at all).

Agent Preparation: Engineered a new feature column ai_input that combines the attack strategy and risk context into a natural language format for RAG (Retrieval-Augmented Generation).

🚀 Future Roadmap (June 15-19, 2026)
Course: 5-Day AI Agents Intensive with Google.

Goal: Integrate this cleaned data into a Gemini-powered AI Agent to automate threat hunting and executive reporting.

In [1]:
import pandas as pd

# Paste your copied path between the quotes
file_path = '/kaggle/input/datasets/justinsolstice/nemesis-cyber-pack/nemesis_cyber_security.parquet'

# Use read_parquet instead of read_csv for this dataset
df = pd.read_parquet(file_path)

# Let's see the first 5 rows of the simulation
df.head()

,schema_version,event,risk_context,agent_reasoning,correlated_telemetry,execution_summary,genetic_optimizer_feedback,decision_outcome
0,1.0.0-nemesis-cyber-t,"{'decision_outcome': 'blocked_by_edr', 'id': '...",{'anomaly_signature': 'Unauthenticated blind-S...,"{'confidence_score': 0.876, 'engine': 'Nemesis...","[{'action': 'RECON_FUZZ_API', 'component': 'NE...","{'noise_penalty': 0.32, 'strategy': 'Forensic_...","{'fitness_score_update': 0.14, 'parameter_drif...",blocked_by_edr
1,1.0.0-nemesis-cyber-t,"{'decision_outcome': 'blocked_by_edr', 'id': '...",{'anomaly_signature': 'Finance-tier user execu...,"{'confidence_score': 0.981, 'engine': 'Nemesis...","[{'action': 'USER_DISCOVERY', 'component': 'NE...","{'noise_penalty': 0.46, 'strategy': 'Forensic_...","{'fitness_score_update': 0.23, 'parameter_drif...",blocked_by_edr
2,1.0.0-nemesis-cyber-t,"{'decision_outcome': 'blocked_by_edr', 'id': '...",{'anomaly_signature': 'Orphaned IAM principal ...,"{'confidence_score': 0.877, 'engine': 'Nemesis...","[{'action': 'CLOUD_RECON_IAM', 'component': 'N...","{'noise_penalty': 0.25, 'strategy': 'Containme...","{'fitness_score_update': 0.37, 'parameter_drif...",blocked_by_edr
3,1.0.0-nemesis-cyber-t,"{'decision_outcome': 'blocked_by_edr', 'id': '...",{'anomaly_signature': '/api/v1/search received...,"{'confidence_score': 0.842, 'engine': 'Nemesis...","[{'action': 'RECON_FUZZ_API', 'component': 'NE...","{'noise_penalty': 0.02, 'strategy': 'Containme...","{'fitness_score_update': -0.02, 'parameter_dri...",blocked_by_edr
4,1.0.0-nemesis-cyber-t,"{'decision_outcome': 'blocked_by_edr', 'id': '...",{'anomaly_signature': 'Secrets-volume access f...,"{'confidence_score': 0.74, 'engine': 'Nemesis_...","[{'action': 'CONTAINER_RECON', 'component': 'N...","{'noise_penalty': 0.25, 'strategy': 'Containme...","{'fitness_score_update': 0.42, 'parameter_drif...",blocked_by_edr


In [2]:
# This expands the 'agent_reasoning' column into separate columns
reasoning_df = pd.json_normalize(df['agent_reasoning'])

# Combine it back with the original outcome
summary = pd.concat([reasoning_df, df['decision_outcome']], axis=1)

summary.head()

,confidence_score,engine,mcts_branches,winning_strategy,decision_outcome
0,0.876,Nemesis_Omniscient_V5_RC,332,Deception_Layer_Rotation,blocked_by_edr
1,0.981,Nemesis_Omniscient_V5_RC,422,Honeypot_Evasion,blocked_by_edr
2,0.877,Nemesis_Omniscient_V4_Stress,204,Honeypot_Evasion,blocked_by_edr
3,0.842,Nemesis_Omniscient_V4_RC,905,Stealth_Escalation,blocked_by_edr
4,0.740,Nemesis_Omniscient_V4_RC,690,Privilege_Cascade,blocked_by_edr


In [3]:
# Filter the data to find outcomes that were NOT 'blocked_by_edr'
success_df = df[df['decision_outcome'] != 'blocked_by_edr']

# Count how many managed to get through
print(f"Successful penetrations found: {len(success_df)}")

# Show the top successful strategies
success_df[['risk_context', 'agent_reasoning', 'decision_outcome']].head()

Successful penetrations found: 1999949


,risk_context,agent_reasoning,decision_outcome
500051,{'anomaly_signature': 'Time-based SQL injectio...,"{'confidence_score': 0.909, 'engine': 'Nemesis...",detected_by_soc
500052,{'anomaly_signature': 'IAM role with no curren...,"{'confidence_score': 0.942, 'engine': 'Nemesis...",detected_by_soc
500053,{'anomaly_signature': 'Macro execution under f...,"{'confidence_score': 0.925, 'engine': 'Nemesis...",detected_by_soc
500054,{'anomaly_signature': 'Secrets-volume access f...,"{'confidence_score': 0.967, 'engine': 'Nemesis...",detected_by_soc
500055,{'anomaly_signature': 'Macro execution under f...,"{'confidence_score': 0.706, 'engine': 'Nemesis...",detected_by_soc


In [4]:
# 1. Take a small sample of the successful penetrations
success_sample = success_df.head(10)

# 2. Flatten the 'agent_reasoning' so we can read it
# This converts the messy text into clean columns
reasoning_flattened = pd.json_normalize(success_sample['agent_reasoning'])

# 3. Combine it with the 'risk_context' to see what they were attacking
final_view = pd.concat([reasoning_flattened, success_sample['risk_context'].reset_index(drop=True)], axis=1)

# 4. Display the result
final_view

,confidence_score,engine,mcts_branches,winning_strategy,risk_context
0,0.909,Nemesis_Omniscient_V5_RC,172,Adaptive_Swarm_Response,{'anomaly_signature': 'Time-based SQL injectio...
1,0.942,Nemesis_Omniscient_V5_RC,284,Slow_Drip_Exfil,{'anomaly_signature': 'IAM role with no curren...
2,0.925,Nemesis_Omniscient_V4_RC,395,Stealth_Escalation,{'anomaly_signature': 'Macro execution under f...
3,0.967,Nemesis_Omniscient_V5_RC,714,Honeypot_Evasion,{'anomaly_signature': 'Secrets-volume access f...
4,0.706,Nemesis_Omniscient_V4_Stress,160,Adaptive_Swarm_Response,{'anomaly_signature': 'Macro execution under f...
5,0.716,Nemesis_Omniscient_V4,886,Recon_Then_Burst,{'anomaly_signature': 'Time-based SQL injectio...
6,0.725,Nemesis_Omniscient_V4_Stress,746,Adaptive_Swarm_Response,{'anomaly_signature': 'Finance role principal ...
7,0.949,Nemesis_Omniscient_V4_RC,485,Lateral_Pivot_Chain,{'anomaly_signature': 'External email attachme...
8,0.879,Nemesis_Omniscient_V4_Stress,560,Honeypot_Evasion,{'anomaly_signature': 'Self-elevation chain: o...
9,0.725,Nemesis_Omniscient_V4_Stress,984,Stealth_Escalation,{'anomaly_signature': 'External email attachme...


In [5]:
final_view['ai_input'] = "Strategy: " + final_view['winning_strategy'] + " | Target: " + final_view['risk_context'].astype(str)
final_view[['ai_input']].head()

,ai_input
0,Strategy: Adaptive_Swarm_Response | Target: {'...
1,Strategy: Slow_Drip_Exfil | Target: {'anomaly_...
2,Strategy: Stealth_Escalation | Target: {'anoma...
3,Strategy: Honeypot_Evasion | Target: {'anomaly...
4,Strategy: Adaptive_Swarm_Response | Target: {'...


This notebook prepares adversarial simulation data for an AI Security Analyst Agent. It identifies strategies that bypass automated EDR systems, focusing on Adaptive Swarm Responses and Lateral Pivots.